# NCN+BGRL

In [24]:
import time
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import copy
import random
from model_contrastive import Encoder, GRACE, drop_feature
from model import CNLinkPredictor, GCN, DropAdj
from NeighborOverlap import train, test
from ogbdataset import loaddataset, randomsplit
from ogb.linkproppred import PygLinkPropPredDataset, Evaluator
from torch.utils.tensorboard import SummaryWriter

from torch.nn.functional import cosine_similarity
from utils_GCA import compute_pr, eigenvector_centrality
from torch.optim import AdamW
from torch_geometric.utils import dropout_adj, to_undirected, degree, k_hop_subgraph
from functional_GCA import drop_edge_weighted, pr_drop_weights, degree_drop_weights, evc_drop_weights, compute_pr, eigenvector_centrality, feature_drop_weights, drop_feature_weighted_2
from bgrl import BGRL, MLP_Predictor, GCN_BGRL,CosineDecayScheduler
from torch_sparse.matmul import spmm_max, spmm_mean, spmm_add


In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

In [3]:
hp = {
    'xdp': 0.7,
    'tdp': 0.3,
    'pt': 0.75,
    'gnnedp': 0.0,
    'preedp': 0.4,
    'predp': 0.05,
    'gnndp': 0.05,
    'probscale': 4.3,
    'proboffset': 2.8,
    'alpha': 1.0,
    'gnnlr': 0.0043,
    'prelr': 0.0024,
    'batch_size': 1152,
    'ln': True,
    'lnnn': True,
    'epochs': 100,
    'runs': 1,
    'hiddim': 256,
    'mplayers': 1,
    'testbs': 8192,
    'maskinput': True,
    'jk': True,
    'use_xlin': True,
    'tailact': True,
}
device = torch.device(f'cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
def legacy_train(epoch, model, predictor, data, split_edge, optimizer, evaluator, hp):
            t1 = time.time()
            loss = train(model, predictor, data, split_edge, optimizer,
                         hp['batch_size'], hp['maskinput'], [], None)
            if epoch % 10 == 0:
                print(f"10 train time {time.time()-t1:.2f} s, loss {loss:.4f}", flush=True)

In [5]:
def legacy_test(run, epoch, model, predictor, data, split_edge, evaluator, bestscore, writer, hp):
    t1 = time.time()
    results, h = test(model, predictor, data, split_edge, evaluator,
                   8192, False)
    print(f"test time {time.time()-t1:.2f} s")
    if bestscore is None:
        bestscore = {key: list(results[key]) for key in results}
    for key, result in results.items():
        writer.add_scalars(f"{key}_{run}", {
            "trn": result[0],
            "val": result[1],
            "tst": result[2]
        }, epoch)
        train_hits, valid_hits, test_hits = result
        if valid_hits > bestscore[key][1]:
            bestscore[key] = list(result)
        print(key)
        print(f'Run: {run + 1:02d}, '
              f'Epoch: {epoch:02d}, '
              f'Train: {100 * train_hits:.2f}%, '
              f'Valid: {100 * valid_hits:.2f}%, '
              f'Test: {100 * test_hits:.2f}%')
    print('---', flush=True)

## Model

In [16]:
class NPredictor(nn.Module):
    def __init__(self,
                 in_channels,
                 hidden_channels,
                 out_channels,
                 num_layers,
                 dropout,
                 edrop=0.0,
                 ln=False,
                 cndeg=-1,
                 use_xlin=False,
                 tailact=False,
                 twolayerlin=False,
                 beta=1.0):
        super().__init__()

        lnfn = lambda dim, ln: nn.LayerNorm(dim) if ln else nn.Identity()

        self.lin = nn.Sequential(nn.Linear(hidden_channels, hidden_channels),
                                 lnfn(hidden_channels, ln),
                                 nn.Dropout(dropout, inplace=True),
                                 nn.ReLU(inplace=True),
                                 nn.Linear(hidden_channels, hidden_channels) if twolayerlin else nn.Identity(),
                                 lnfn(hidden_channels, ln) if twolayerlin else nn.Identity(),
                                 nn.Dropout(dropout, inplace=True) if twolayerlin else nn.Identity(),
                                 nn.ReLU(inplace=True) if twolayerlin else nn.Identity(),
                                 nn.Linear(hidden_channels, out_channels))

    def get_neighbors(self, edge_index, node_idx):
            mask = edge_index[0] == node_idx
            neighbors = edge_index[1][mask]
            return neighbors
    
    def forward(self,
                           h,
                           node_id,
                           edge_index):
        x = h[node_id]
        neight = self.get_neighbors(edge_index, node_id)
        xs = x
        for n in neight:
            xs = xs + h[n]
        return self.lin(xs)

In [38]:
class BGRL_NCN_PRED(torch.nn.Module):
    r"""BGRL architecture for Graph representation learning.

    Args:
        encoder (torch.nn.Module): Encoder network to be duplicated and used in both online and target networks.
        predictor (torch.nn.Module): Predictor network used to predict the target projection from the online projection.

    .. note::
        `encoder` must have a `reset_parameters` method, as the weights of the target network will be initialized
        differently from the online network.
    """
    def __init__(self, encoder, predictor):
        super().__init__()
        # online network
        self.online_encoder = encoder
        self.predictor = predictor

        # target network
        self.target_encoder = copy.deepcopy(encoder)

        # reinitialize weights
        self.target_encoder.reset_parameters()
        # stop gradient
        for param in self.target_encoder.parameters():
            param.requires_grad = False

    def trainable_parameters(self):
        r"""Returns the parameters that will be updated via an optimizer."""
        return list(self.online_encoder.parameters()) + list(self.predictor.parameters())

    @torch.no_grad()
    def update_target_network(self, mm):
        r"""Performs a momentum update of the target network's weights.

        Args:
            mm (float): Momentum used in moving average update.
        """
        assert 0.0 <= mm <= 1.0, "Momentum needs to be between 0.0 and 1.0, got %.5f" % mm
        for param_q, param_k in zip(self.online_encoder.parameters(), self.target_encoder.parameters()):
            param_k.data.mul_(mm).add_(param_q.data, alpha=1. - mm)
            # mm c'est le poids de la target ~= param_k.data[i] = param_k.data[i] * mm + param_q.data[i] * (1 - mm)

    def forward(self, online_x, target_x):
        # forward online network
        online_y = self.online_encoder(online_x.x, online_x.edge_index)
        online_q = torch.empty_like(online_y)
        node_id = random.sample(range(len(online_y)),  int(len(online_y)/100))
        for it in node_id:
            online_q[it] = self.predictor(online_y, it, online_x.edge_index)

        # forward target network
        with torch.no_grad():
            target_y = self.target_encoder(target_x.x, target_x.edge_index).detach()
        return online_q, target_y


## Run

In [39]:
def run_bgrl(r, dataset, evaluator, hp):
    writer = SummaryWriter(f"./rec/BGRL_NCN")
    writer.add_text("hyperparams", str(hp))
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
    bestscore = None
    # build model
    basic_encoder = GCN_BGRL([data.num_features, hp['hiddim']], batchnorm=True).to(device)
    predictor = NPredictor(hp['hiddim'], hp['hiddim'], hp['hiddim'], 3,
                       hp['predp'], hp['preedp'], hp['lnnn']).to(device)
    model = BGRL_NCN_PRED(basic_encoder, predictor).to(device)
    pretrain_bgrl(model, data)
    predictor = CNLinkPredictor(hp['hiddim'], hp['hiddim'], 1, 3,
                       hp['predp'], hp['preedp'], hp['lnnn']).to(device)
    optimizer = torch.optim.Adam([{'params': model.parameters(), "lr": hp['gnnlr']}, 
       {'params': predictor.parameters(), 'lr': hp['prelr']}])

    for epoch in range(1, 1 + hp['epochs']):
        legacy_train(epoch, basic_encoder, predictor, data, split_edge, optimizer, evaluator, hp)
        print("epoch ", epoch)
        if epoch % 100 == 0:
            legacy_test(r, epoch, basic_encoder, predictor, data, split_edge, evaluator, bestscore, writer, hp)

In [40]:
def pretrain_bgrl(model, data):
    param = {
        'learning_rate': 0.01,
        'num_hidden': 256,
        'num_proj_hidden': 32,
        'activation': 'prelu',
        'base_model': 'GCNConv',
        'num_layers': 2,
        'drop_edge_rate_1': 0.3,
        'drop_edge_rate_2': 0.4,
        'drop_feature_rate_1': 0.1,
        'drop_feature_rate_2': 0.0,
        'tau': 0.4,
        'num_epochs': 2000,
        'weight_decay': 1e-5,
        'drop_scheme': 'degree',
    }
      # optimizer
    optimizer = AdamW(model.trainable_parameters(), lr=param['learning_rate'], weight_decay=param['weight_decay'])

    # scheduler
    lr_scheduler = CosineDecayScheduler(param['learning_rate'], 1000, param['num_epochs'])
    mm_scheduler = CosineDecayScheduler(1 - 0.99, 0, param['num_epochs'])

    t1 = time.time()
    for epoch in range(1, param['num_epochs'] + 1):
        model.train()

        lr = lr_scheduler.get(epoch)
        mm = 1 - mm_scheduler.get(epoch)

        
        optimizer.zero_grad()
        data_c1 = data.clone()
        data_c2 = data.clone()
        data_c1.edge_index = dropout_adj(data.edge_index, p=param[f'drop_edge_rate_{1}'])[0]
        data_c2.edge_index = dropout_adj(data.edge_index, p=param[f'drop_edge_rate_{2}'])[0]
        
        data_c1.x = drop_feature(data.x, param['drop_feature_rate_1'])
        data_c2.x = drop_feature(data.x, param['drop_feature_rate_2'])

        z1, y2 = model(data_c1, data_c2)
        z2, y1 = model(data_c2, data_c1)

        loss = 2 - cosine_similarity(z1, y2.detach(), dim=-1).mean() - cosine_similarity(z2, y1.detach(), dim=-1).mean() # loss simple
        loss.backward()
        optimizer.step()
        model.update_target_network(mm)
        if epoch % 100 == 0:
            print(f'(T) | Epoch={epoch:03d}, loss={loss:.4f}')
    print(f"pretrain time {time.time()-t1:.2f} s, loss {loss:.4f}", flush=True)

In [41]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    print(f'################### {dataset} #################')
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
    else:
        evaluator = Evaluator(name=f'ogbl-{args.dataset}')
    for r in range(hp['runs']):
        set_seed(r)
        run_bgrl(r, dataset, evaluator, hp)

################### Cora #################


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


(T) | Epoch=100, loss=2.0532
(T) | Epoch=200, loss=2.1260
(T) | Epoch=300, loss=1.8938
(T) | Epoch=400, loss=2.2454
(T) | Epoch=500, loss=2.2595
(T) | Epoch=600, loss=2.2302
(T) | Epoch=700, loss=2.2727
(T) | Epoch=800, loss=2.2701
(T) | Epoch=900, loss=2.2240
(T) | Epoch=1000, loss=2.2614
(T) | Epoch=1100, loss=2.2565
(T) | Epoch=1200, loss=2.2551
(T) | Epoch=1300, loss=2.2500
(T) | Epoch=1400, loss=2.2546
(T) | Epoch=1500, loss=2.2491
(T) | Epoch=1600, loss=2.2437
(T) | Epoch=1700, loss=2.2448
(T) | Epoch=1800, loss=2.2457
(T) | Epoch=1900, loss=2.2444
(T) | Epoch=2000, loss=2.2470
pretrain time 167.90 s, loss 2.2470
epoch  1
epoch  2
epoch  3
epoch  4
epoch  5
epoch  6
epoch  7
epoch  8
epoch  9
10 train time 0.10 s, loss 1.0862
epoch  10
epoch  11
epoch  12
epoch  13
epoch  14
epoch  15
epoch  16
epoch  17
epoch  18
epoch  19
10 train time 0.07 s, loss 0.3149
epoch  20
epoch  21
epoch  22
epoch  23
epoch  24
epoch  25
epoch  26
epoch  27
epoch  28
epoch  29
10 train time 0.06 s, lo

/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


3327 tensor(3325)
dataset split 
train edge 3187
valid edge 455
valid edge_neg 910
test edge 910
test edge_neg 910


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


(T) | Epoch=100, loss=1.1291
(T) | Epoch=200, loss=1.2585
(T) | Epoch=300, loss=2.1055


KeyboardInterrupt: 